#### different sampling resolutions for different datasets
#### maybe seasonality, maybe monthly vs daily aggregations

In [ ]:
import pyEDM
import pandas as pd
import numpy as np
import random
from collections import Counter

p22 = pd.read_csv('./data/processed_22.csv')
p23 = pd.read_csv('./data/processed_23.csv')
p24 = pd.read_csv('./data/processed_24.csv')
p25 = pd.read_csv('./data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [17]:
# top 20 features from paper + target variable
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    "Solidity", "texture_uniformity", "texture_smoothness",
    "texture_average_gray_level", "texture_entropy",
    "texture_average_contrast", "H90", "H180", "Hflip",
    "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df = df.sort_values("date").set_index("date")
df = df.resample("D").mean()

df_filled = df.copy()

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)

df_filled = df_filled.resample("QS-DEC").mean() 

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    df_norm[col] = (df_filled[col] - mu) / sigma

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)
df_mv = df_mv[["t"] + features]

target = "Lpoly_expected_ml"
predictors = [col for col in features if col != target]

df_mv.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,texture_average_gray_level,texture_entropy,texture_average_contrast,H90,H180,Hflip,Extent,EquivDiameter,ConvexArea,ConvexPerimeter
0,1,0.006919,0.903315,0.833753,0.802524,0.991661,0.959840,0.801293,-1.245279,0.420515,...,-1.042663,0.883462,-0.977959,-1.300742,0.500508,0.755955,0.678541,0.943737,0.910009,0.913805
1,2,-0.406765,1.309002,1.416491,1.072393,1.226683,1.281819,1.041074,-1.338694,0.188918,...,-1.026531,0.814787,-0.976874,-1.144693,0.876719,1.143226,0.544346,1.182606,1.324579,1.164783
2,3,-0.754362,-0.553252,-0.513203,-0.194881,-0.628318,-0.297180,2.145521,1.148678,-2.947565,...,0.325865,-1.416794,-0.429175,2.121569,0.834069,0.015581,-2.702235,-0.503091,-0.492288,-0.374941
3,4,0.590322,0.406472,0.264470,0.565746,0.511079,0.657806,1.235308,-0.400891,-0.637295,...,-0.647596,0.380335,-0.847181,0.018309,1.159382,0.953342,-0.506693,0.535834,0.454874,0.570876
4,5,0.055686,0.618333,0.576810,0.579966,0.674944,0.525888,-0.009149,-0.798773,1.010705,...,-0.640168,0.715174,-0.330148,-0.733128,0.232534,0.415915,0.932742,0.638836,0.607446,0.609819


In [ ]:
# # top 20 features from paper + target variable
# features = [
#     "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
#     "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
#     "Solidity", "texture_uniformity", "texture_smoothness",
#     "texture_average_gray_level", "texture_entropy",
#     "texture_average_contrast", "H90", "H180", "Hflip",
#     "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
# ]

# df = combined[["date"] + features].copy()
# df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
# df = df.sort_values("date").set_index("date")
# df = df.resample("MS").mean()

# df_filled = df.copy()

# for col in features:
#     ema = df[col].ewm(span=6, adjust=False).mean() # 6-month EMA for monthly data
#     df_filled[col] = df[col].fillna(ema)

# # normalize features
# df_norm = df_filled.copy()
# for col in features:
#     mu = df_filled[col].mean()
#     sigma = df_filled[col].std()
#     df_norm[col] = (df_filled[col] - mu) / sigma

# df_mv = df_norm.copy()
# df_mv = df_mv.reset_index()
# df_mv["t"] = np.arange(1, len(df_mv) + 1)
# df_mv = df_mv[["t"] + features]

# target = "Lpoly_expected_ml"
# predictors = [col for col in features if col != target]

# df_mv.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,texture_average_gray_level,texture_entropy,texture_average_contrast,H90,H180,Hflip,Extent,EquivDiameter,ConvexArea,ConvexPerimeter
0,1,-0.074691,0.668402,0.560731,0.623628,0.799668,0.733229,0.831920,-1.050448,0.569475,...,-0.910792,0.818111,-0.924692,-1.079816,0.189038,0.446730,0.790940,0.762041,0.665600,0.719807
1,2,-0.092317,0.836251,0.790148,0.740426,0.891987,0.895198,0.262627,-1.088902,0.026154,...,-0.870239,0.405242,-0.814176,-0.938895,0.496643,0.803889,0.246293,0.841999,0.859137,0.837133
2,3,-0.366731,1.524992,1.705260,1.251874,1.351711,1.505533,1.552370,-1.290263,0.082905,...,-1.052208,0.475486,-1.064016,-0.943300,1.073525,1.346107,0.415003,1.324565,1.562601,1.323837
3,4,-0.161728,0.947397,0.934559,0.826963,0.976235,0.986942,0.921581,-1.126814,0.319573,...,-0.942921,0.635959,-0.941945,-1.012052,0.504525,0.776584,0.572377,0.939080,0.961384,0.916331
4,5,-0.161728,0.947397,0.934559,0.826963,0.976235,0.986942,0.921581,-1.126814,0.319573,...,-0.942921,0.635959,-0.941945,-1.012052,0.504525,0.776584,0.572377,0.939080,0.961384,0.916331


In [21]:
def one_simplex(df, target, features, E=4, Tp=1):
    # Randomly select 3 features (+ the target) for the simplex projection
    chosen_features = random.sample(features, 3)
    columns = [target] + chosen_features
    columns_str = " ".join(columns) # has to be 'space separated' idk ????

    N = len(df)
    res = pyEDM.Simplex(
        dataFrame=df,
        columns=columns_str,
        target=target,
        E=E,
        tau=1,
        Tp=Tp,
        lib=f"1 {N}",
        pred=f"1 {N}"
    )

    obs = res["Observations"].to_numpy()
    pred = res["Predictions"].to_numpy()

    mask = np.isfinite(obs) & np.isfinite(pred)
    obs = obs[mask]
    pred = pred[mask]

    if len(obs) < 10 or np.std(obs) == 0 or np.std(pred) == 0:
        return np.nan, chosen_features

    rho = np.corrcoef(obs, pred)[0, 1]
    rmse = np.sqrt(np.mean((obs - pred) ** 2))
    mae = np.mean(np.abs(obs - pred))
    return rho, rmse, mae, chosen_features

def multiview_big(df, target, features, Tp, n_trials=500):
    results = []

    for i in range(n_trials):
        rho, rmse, mae, chosen = one_simplex(df, target, features, E=4, Tp=Tp)

        results.append({
            "rho": rho,
            "rmse": rmse,
            "mae": mae,
            "features": chosen
        })

    return pd.DataFrame(results)

def multiview_yes(df_mv, target, predictors):
    # wrapper main function
    x = df_mv[target].to_numpy()

    summary_rows = []
    feature_importance_by_tp = {}
    
    for Tp in range(1, 3): # 1 to seasons ahead
        mv = multiview_big(df_mv, target, predictors, Tp, n_trials=500)

        acf = abs(pd.Series(x).autocorr(lag=Tp))
        mv["rho_eff"] = mv["rho"] - acf

        # summary stats
        summary_rows.append({
            "Tp": Tp,
            "rho_eff_mean": mv["rho_eff"].mean(),
            "rmse_mean": mv["rmse"].mean(),
            "mae_mean": mv["mae"].mean(),
            "n_models": len(mv)
        })

        # feature importance (top 5%)
        top = mv.nlargest(int(0.05 * len(mv)), "rho_eff")

        counts = Counter()
        for feats in top["features"]:
            for f in feats:
                counts[f] += 1

        importance = pd.Series(counts) / len(top)
        feature_importance_by_tp[Tp] = importance.sort_values(ascending=False)
    
    return summary_rows, feature_importance_by_tp

In [13]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_mv, target, predictors)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.885827   0.624077  0.311823       500
1    2      0.759515   0.722469  0.344097       500
2    3      0.781477   0.757149  0.344137       500
3    4     -0.086275   1.153268  0.676182       500
4    5     -0.353003   1.312449  0.875473       500
5    6     -0.604440   1.287481  0.784605       500
6    7     -0.318442   1.289588  0.761703       500
7    8     -0.272119   1.279602  0.771360       500
8    9     -0.272406   1.278586  0.835910       500
9   10      0.052497   1.289731  0.947438       500
10  11     -0.284618   1.396305  0.959044       500
11  12     -0.324312   1.411924  0.940076       500


In [22]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_mv, target, predictors)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

   Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0   1      0.443901   0.944236  0.783281       500
1   2      0.022650   0.995244  0.838107       500
